# NB01 — Data Ingestion & Exploratory Data Analysis
## BAREKENG MS 21575 — A Comparative Analysis of Isolation Forest and Local Outlier Factor

**Stage overview.** Entry point of the pipeline. Loads the raw Medicare provider claims
extract, converts the seven numerical features to float, characterises their distributions,
and emits the model-ready feature matrix consumed by NB02.

| | |
|---|---|
| **Manuscript link** | §2.1 Dataset and Features; Figure 1; Figure 2 |
| **Reviewer comments** | Supports **E1** (distribution analysis / mathematical depth) |
| **Upstream** | none — this is the first notebook |
| **Downstream** | NB02 (`02_baseline_models.ipynb`) |

### Inputs
| Artifact | Path |
|---|---|
| Raw dataset | `data/raw/healthcare_providers.csv` (100,000 × 27) |

### Outputs
| Artifact | Path |
|---|---|
| Feature matrix | `data/processed/features_raw.parquet` (100,000 × 7) |
| Figure 1 | `outputs/figures/nb01_fig1_distributions.png` (300 dpi) |
| Figure 2 | `outputs/figures/nb01_fig2_correlation_heatmap.png` (300 dpi) |
| Descriptive stats | `outputs/tables/nb01_descriptive_statistics.csv` |
| Handshake | `outputs/notebook_exports/summary_NB01.json` |

---

> ### ⚠️ The one thing this notebook must get right
>
> The seven numerical columns are stored as `object` because values of 1,000 and above
> carry a **thousands separator**. They must be stripped *before* numeric conversion.
>
> Converting with `pd.to_numeric(..., errors='coerce')` and no strip turns every value
> ≥ 1,000 into `NaN` — 12,962 of them. Because the loss lands entirely in the upper tail,
> median-imputing it rewrites a 282,739-service provider as 43 services and a \$62,694
> charge as \$146, erasing precisely the records the anomaly detectors exist to find.
> This destroyed the first rebuild of this pipeline.
>
> **Cell 05 strips first, then asserts zero NaN and halts on failure. Never relax that
> assertion — if it fires, diagnose the input.** See `REPRODUCIBILITY.md` §4.1.

In [1]:
# Cell 01 — Mount Storage & Define Paths
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PROJECT_FOLDER_NAME = "BarekengPaper1-Revision"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = Path(f"/content/drive/MyDrive/{PROJECT_FOLDER_NAME}")
else:
    BASE = Path.cwd()
    if BASE.name == "notebooks":
        BASE = BASE.parent

RAW = BASE / "data" / "raw"
INTERIM = BASE / "data" / "interim"
PROCESSED = BASE / "data" / "processed"
FIGURES = BASE / "outputs" / "figures"
TABLES = BASE / "outputs" / "tables"
MODELS = BASE / "outputs" / "models"
EXPORTS = BASE / "outputs" / "notebook_exports"
MANUSCRIPT = BASE / "outputs" / "manuscript_ready"

ALL_DIRS = [RAW, INTERIM, PROCESSED, FIGURES, TABLES, MODELS, EXPORTS, MANUSCRIPT]

created = [d for d in ALL_DIRS if not d.exists()]
for d in created:
    d.mkdir(parents=True, exist_ok=True)

if not BASE.exists():
    raise FileNotFoundError(f"Project root not found: {BASE}")

print(f"Environment : {'Google Colab' if IN_COLAB else 'Local workstation'}")
print(f"Base        : {BASE}")
print(f"Directories : {len(created)} created, {len(ALL_DIRS) - len(created)} already present")

Mounted at /content/drive
Environment : Google Colab
Base        : /content/drive/MyDrive/BarekengPaper1-Revision
Directories : 1 created, 7 already present


In [2]:
# Cell 02 — Imports, Global Seed, Publication Style & Environment Capture

import hashlib
import json
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.titlesize": 12,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# Recorded into the handshake so requirements.txt can be pinned from an observed
# environment rather than a guessed one. Parity for Tables 1-4 was demonstrated on
# numpy 2.0.2 / pandas 2.3.3 / scikit-learn 1.6.1 / scipy 1.13.1.
ENVIRONMENT = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
}

print(f"Seed fixed to {SEED}. Publication style applied.")
for pkg, ver in ENVIRONMENT.items():
    print(f"  {pkg:<12} {ver}")

Seed fixed to 42. Publication style applied.
  python       3.13.15
  numpy        2.1.3
  pandas       2.2.3
  matplotlib   3.10.0
  seaborn      0.13.2


In [3]:
# Cell 03 — Verify: Raw Data Presence & SHA-256 Provenance
# NB01 has no upstream handshake; this cell is its equivalent integrity gate.

RAW_FILE = RAW / "healthcare_providers.csv"
EXPECTED_SHA256 = "3ab905111f47b32ab030ed5b106bbaf602961ed04d4a16ff615d2453911b9991"
EXPECTED_BYTES = 23_362_829

if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at {RAW_FILE}\n"
        f"Place healthcare_providers.csv in {RAW} before running this notebook."
    )

digest = hashlib.sha256()
with open(RAW_FILE, "rb") as fh:
    for chunk in iter(lambda: fh.read(1024 * 1024), b""):
        digest.update(chunk)
RAW_SHA256 = digest.hexdigest()
RAW_BYTES = RAW_FILE.stat().st_size
CHECKSUM_OK = RAW_SHA256 == EXPECTED_SHA256

print(f"File   : {RAW_FILE}")
print(f"Size   : {RAW_BYTES:,} bytes (expected {EXPECTED_BYTES:,})")
print(f"SHA-256: {RAW_SHA256}")

if CHECKSUM_OK:
    print("\nChecksum matches the snapshot used for parity verification.")
else:
    # A warning, not a halt: a different snapshot may still be legitimate, but every
    # parity anchor in REPRODUCIBILITY.md section 5 was verified against this exact file.
    print(
        "\n*** WARNING: checksum mismatch ***\n"
        f"  expected {EXPECTED_SHA256}\n"
        f"  observed {RAW_SHA256}\n"
        "This is NOT the snapshot the published Tables 1-4 were reproduced from.\n"
        "Parity failures downstream should be attributed to the data, not the models.\n"
        "Investigate before trusting any result from this run."
    )

File   : /content/drive/MyDrive/BarekengPaper1-Revision/data/raw/healthcare_providers.csv
Size   : 23,362,829 bytes (expected 23,362,829)
SHA-256: 3ab905111f47b32ab030ed5b106bbaf602961ed04d4a16ff615d2453911b9991

Checksum matches the snapshot used for parity verification.


---
## Section 1 — Ingestion & Numeric Conversion

In [5]:
# Cell 04 — Load: Read Raw Claims Extract

df_raw = pd.read_csv(RAW_FILE, low_memory=False)

assert df_raw.shape == (100_000, 27), (
    f"Unexpected raw shape {df_raw.shape}; manuscript §2.1 specifies (100000, 27)."
)

print(f"Loaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

Loaded 100,000 rows x 27 columns


,index,National Provider Identifier,Last Name/Organization Name of the Provider,First Name of the Provider,Middle Initial of the Provider,Credentials of the Provider,Gender of the Provider,Entity Type of the Provider,Street Address 1 of the Provider,Street Address 2 of the Provider,...,HCPCS Code,HCPCS Description,HCPCS Drug Indicator,Number of Services,Number of Medicare Beneficiaries,Number of Distinct Medicare Beneficiary/Per Day Services,Average Medicare Allowed Amount,Average Submitted Charge Amount,Average Medicare Payment Amount,Average Medicare Standardized Amount
0,8774979,1891106191,UPADHYAYULA,SATYASREE,NaN,M.D.,F,I,1402 S GRAND BLVD,FDT 14TH FLOOR,...,99223,"Initial hospital inpatient care, typically 70 ...",N,27,24,27,200.58777778,305.21111111,157.26222222,160.90888889
1,3354385,1346202256,JONES,WENDY,P,M.D.,F,I,2950 VILLAGE DR,NaN,...,G0202,"Screening mammography, bilateral (2-view study...",N,175,175,175,123.73,548.8,118.83,135.31525714
2,3001884,1306820956,DUROCHER,RICHARD,W,DPM,M,I,20 WASHINGTON AVE,STE 212,...,99348,"Established patient home visit, typically 25 m...",N,32,13,32,90.65,155,64.4396875,60.5959375
3,7594822,1770523540,FULLARD,JASPER,NaN,MD,M,I,5746 N BROADWAY ST,NaN,...,81002,"Urinalysis, manual test",N,20,18,20,3.5,5,3.43,3.43
4,746159,1073627758,PERROTTI,ANTHONY,E,DO,M,I,875 MILITARY TRL,SUITE 200,...,96372,Injection beneath the skin or into muscle for ...,N,33,24,31,26.52,40,19.539393939,19.057575758


In [6]:
# Cell 05 — Transform: Extract 7 Features, Strip Currency Formatting, Assert Integrity
# CRITICAL CELL. See the warning in Cell 00 and REPRODUCIBILITY.md section 4.1.

FEATURES = [
    "Number of Services",
    "Number of Medicare Beneficiaries",
    "Number of Distinct Medicare Beneficiary/Per Day Services",
    "Average Medicare Allowed Amount",
    "Average Submitted Charge Amount",
    "Average Medicare Payment Amount",
    "Average Medicare Standardized Amount",
]

missing_cols = [c for c in FEATURES if c not in df_raw.columns]
if missing_cols:
    raise KeyError(f"Columns absent from raw file: {missing_cols}")


def clean_numeric(series):
    # Ported verbatim from the original notebook's cell [3]. The strip MUST precede
    # conversion: without it every value >= 1,000 becomes NaN.
    return (
        series.astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .replace("nan", np.nan)
        .astype(float)
    )


df_features = df_raw[FEATURES].copy()
dtypes_before = df_features.dtypes.astype(str).to_dict()

for col in FEATURES:
    df_features[col] = clean_numeric(df_features[col])

nan_counts = df_features.isna().sum()
NAN_TOTAL = int(nan_counts.sum())

if NAN_TOTAL > 0:
    offenders = nan_counts[nan_counts > 0]
    report = "\n".join(f"    {c}: {n:,}" for c, n in offenders.items())
    sample_col = offenders.index[0]
    sample_vals = df_raw.loc[df_features[sample_col].isna(), sample_col].head(5).tolist()
    raise ValueError(
        f"Currency stripping failed: {NAN_TOTAL:,} NaN produced.\n"
        f"{report}\n"
        f"  Sample unparsed values from '{sample_col}': {sample_vals}\n\n"
        "DO NOT impute these and DO NOT relax this assertion. A non-zero count here\n"
        "means the raw format differs from the verified snapshot -- the NaNs will be\n"
        "concentrated in the upper tail and imputing them destroys the anomaly signal.\n"
        "See REPRODUCIBILITY.md section 4.1."
    )

# Retained for fidelity to the original notebook's cell [4]. Verified no-op: the
# assertion above guarantees there is nothing to fill.
df_features = df_features.fillna(df_features.median())

assert df_features.shape == (100_000, 7), f"Unexpected feature shape {df_features.shape}"
assert (df_features.dtypes == "float64").all(), "All 7 features must be float64"

print(f"Converted {len(FEATURES)} features to float64. NaN produced: {NAN_TOTAL}")
print(f"Matrix shape: {df_features.shape}\n")
print("Object -> float64 conversion:")
for col in FEATURES:
    print(f"  {dtypes_before[col]:<8} -> float64   {col}")

Converted 7 features to float64. NaN produced: 0
Matrix shape: (100000, 7)

Object -> float64 conversion:
  object   -> float64   Number of Services
  object   -> float64   Number of Medicare Beneficiaries
  object   -> float64   Number of Distinct Medicare Beneficiary/Per Day Services
  object   -> float64   Average Medicare Allowed Amount
  object   -> float64   Average Submitted Charge Amount
  object   -> float64   Average Medicare Payment Amount
  object   -> float64   Average Medicare Standardized Amount


In [7]:
# Cell 06 — Feature: Descriptive Statistics & Distributional Diagnostics
# Quantitative basis for the manuscript's distribution analysis (E1).

desc = df_features.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T
desc["skewness"] = df_features.skew()
desc["kurtosis"] = df_features.kurtosis()
desc["max_median_ratio"] = desc["max"] / desc["50%"]
desc = desc.rename(columns={"50%": "median"})

DESCRIPTIVE_STATS = desc.round(4)

print("Descriptive statistics (7 numerical features, n = 100,000)\n")
print(DESCRIPTIVE_STATS[["mean", "std", "min", "median", "95%", "max", "skewness"]].to_string())
print(
    f"\nSkewness spans {desc['skewness'].min():.1f} to {desc['skewness'].max():.1f}; "
    f"all seven features are severely right-skewed.\n"
    "Two consequences carried forward:\n"
    "  1. Figure 1 uses log-scaled axes -- a linear axis renders these unreadable.\n"
    "  2. Non-parametric tests (Mann-Whitney U) are required in NB06; normality-assuming\n"
    "     tests are not defensible on these distributions."
)

Descriptive statistics (7 numerical features, n = 100,000)

                                                              mean        std      min    median        95%         max  skewness
Number of Services                                        239.6714  2493.1871  11.0000   43.0000   608.0000  282739.000   54.4379
Number of Medicare Beneficiaries                           89.8093  1109.6169  11.0000   32.0000   269.0000  190306.000  124.1265
Number of Distinct Medicare Beneficiary/Per Day Services  142.1157  1640.2272  11.0000   40.0000   484.0000  282737.000  120.0429
Average Medicare Allowed Amount                           101.4342   257.2428   0.0100   65.0950   248.7312   20494.000   31.6725
Average Submitted Charge Amount                           354.5505  1062.6083   0.0100  146.0000  1251.7997   62694.000   18.3090
Average Medicare Payment Amount                            77.3588   199.7188   0.0087   47.0202   192.9547   16067.300   31.6515
Average Medicare Standardized 

---
## Section 2 — Publication Figures

**Figure 1** replaces the version in the submitted manuscript, which plotted a single
feature as a dense categorical bar chart with several hundred unreadable axis labels.
Given skewness between 18 and 124, log-scaled histograms across all seven features
communicate the distributional structure the editor asked to see.

In [ ]:
# Cell 07 — Plot: Figure 1, Feature Distributions (log-scaled)

fig, axes = plt.subplots(4, 2, figsize=(11, 13))
axes = axes.ravel()

for ax, col in zip(axes, FEATURES):
    series = df_features[col]
    bins = np.logspace(np.log10(series.min()), np.log10(series.max()), 60)

    ax.hist(series, bins=bins, color="#4C72B0", edgecolor="white", linewidth=0.3)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.axvline(series.median(), color="#C44E52", linestyle="--", linewidth=1.2,
               label=f"median = {series.median():,.2f}")
    ax.axvline(series.mean(), color="#DD8452", linestyle=":", linewidth=1.2,
               label=f"mean = {series.mean():,.2f}")
    ax.set_title(col, fontsize=9.5)
    ax.set_xlabel("value (log scale)")
    ax.set_ylabel("frequency (log scale)")
    ax.legend(fontsize=7.5, frameon=False)

axes[-1].axis("off")
axes[-1].text(
    0.02, 0.5,
    "All seven features are severely\n"
    f"right-skewed (skewness {desc['skewness'].min():.0f}–{desc['skewness'].max():.0f}).\n\n"
    "Both axes are logarithmic: the extreme\n"
    "upper tails that motivate unsupervised\n"
    "anomaly detection are invisible on a\n"
    "linear scale.",
    fontsize=9, va="center", family="serif",
)

fig.suptitle("Figure 1. Distributions of the Seven Numerical Features (n = 100,000)", y=0.995)
fig.tight_layout()

fig1_png = FIGURES / "nb01_fig1_distributions.png"
fig.savefig(fig1_png, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved Figure 1 -> {fig1_png.name} (300 dpi)")

In [ ]:
# Cell 08 — Plot: Figure 2, Feature Correlation Heatmap

CORRELATION = df_features.corr(method="pearson")
mask = np.triu(np.ones_like(CORRELATION, dtype=bool), k=1)

short_labels = [
    "No. Services",
    "No. Beneficiaries",
    "Distinct Benef./Day",
    "Avg Allowed Amt",
    "Avg Submitted Chg",
    "Avg Medicare Pymt",
    "Avg Standardized Amt",
]

fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(
    CORRELATION, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.75, "label": "Pearson $r$"},
    xticklabels=short_labels, yticklabels=short_labels, ax=ax,
)
ax.set_title("Figure 2. Pearson Correlation of Numerical Features", pad=12)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)
fig.tight_layout()

fig2_png = FIGURES / "nb01_fig2_correlation_heatmap.png"
fig.savefig(fig2_png, dpi=300, bbox_inches="tight")
plt.show()

offdiag = CORRELATION.where(~np.eye(len(CORRELATION), dtype=bool))
print(f"Saved Figure 2 -> {fig2_png.name} (300 dpi)")
print(f"Strongest off-diagonal correlation: {offdiag.max().max():.4f}")
print(f"Weakest  off-diagonal correlation: {offdiag.min().min():.4f}")

---
## Section 3 — Persistence & Handover

In [10]:
# Cell 09 — Export: Feature Matrix & Descriptive Statistics Table

features_path = PROCESSED / "features_raw.parquet"
df_features.to_parquet(features_path, index=False)

stats_path = TABLES / "nb01_descriptive_statistics.csv"
DESCRIPTIVE_STATS.to_csv(stats_path, index_label="feature")

corr_path = TABLES / "nb01_correlation_matrix.csv"
CORRELATION.round(4).to_csv(corr_path, index_label="feature")

roundtrip = pd.read_parquet(features_path)
assert roundtrip.shape == df_features.shape, "Parquet round-trip changed shape"
assert roundtrip.isna().sum().sum() == 0, "Parquet round-trip introduced NaN"

print(f"{features_path.relative_to(BASE)}  {df_features.shape}")
print(f"{stats_path.relative_to(BASE)}")
print(f"{corr_path.relative_to(BASE)}")
print("\nParquet round-trip verified: shape preserved, no NaN introduced.")

data/processed/features_raw.parquet  (100000, 7)
outputs/tables/nb01_descriptive_statistics.csv
outputs/tables/nb01_correlation_matrix.csv

Parquet round-trip verified: shape preserved, no NaN introduced.


In [11]:
# Cell 10 — Export: JSON Handshake (NB01 -> NB02)

summary = {
    "notebook": "NB01",
    "status": "SUCCESS",
    "timestamp": pd.Timestamp.now().isoformat(),
    "environment": ENVIRONMENT,
    "seed": SEED,
    "inputs_consumed": ["data/raw/healthcare_providers.csv"],
    "data_provenance": {
        "sha256": RAW_SHA256,
        "bytes": int(RAW_BYTES),
        "checksum_matches_parity_snapshot": bool(CHECKSUM_OK),
    },
    "outputs": {
        "features_matrix": "data/processed/features_raw.parquet",
        "descriptive_stats": "outputs/tables/nb01_descriptive_statistics.csv",
        "correlation_matrix": "outputs/tables/nb01_correlation_matrix.csv",
        "figure_1": "outputs/figures/nb01_fig1_distributions.png",
        "figure_2": "outputs/figures/nb01_fig2_correlation_heatmap.png",
    },
    "key_results": {
        "raw_shape": list(df_raw.shape),
        "feature_shape": list(df_features.shape),
        "feature_names": FEATURES,
        "nan_after_cleaning": NAN_TOTAL,
        "median": {c: float(df_features[c].median()) for c in FEATURES},
        "mean": {c: float(df_features[c].mean()) for c in FEATURES},
        "std": {c: float(df_features[c].std()) for c in FEATURES},
        "min": {c: float(df_features[c].min()) for c in FEATURES},
        "max": {c: float(df_features[c].max()) for c in FEATURES},
        "skewness": {c: float(df_features[c].skew()) for c in FEATURES},
        "max_abs_offdiag_correlation": float(
            CORRELATION.where(~np.eye(len(CORRELATION), dtype=bool)).abs().max().max()
        ),
    },
    "assertions": {
        "raw_shape_is_100000x27": list(df_raw.shape) == [100000, 27],
        "feature_shape_is_100000x7": list(df_features.shape) == [100000, 7],
        "zero_nan_after_currency_strip": NAN_TOTAL == 0,
        "all_features_float64": bool((df_features.dtypes == "float64").all()),
        "parquet_roundtrip_verified": True,
    },
    "downstream_note": (
        "NB02 consumes data/processed/features_raw.parquet. Apply StandardScaler to all "
        "7 features, then fit IsolationForest(n_estimators=100, contamination=0.05, "
        "random_state=42) and LocalOutlierFactor(n_neighbors=20) at contamination 'auto' "
        "and 0.05, all on the scaled matrix. Expected: IF 5000, LOF(0.05) 5000, "
        "LOF(auto) 2565 -- the last of these is the first diagnostic parity check in the "
        "pipeline; the two 5000s are contamination x n and hold regardless of input."
    ),
}

summary_path = EXPORTS / "summary_NB01.json"
with open(summary_path, "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2)

failed = [k for k, v in summary["assertions"].items() if not v]
if failed:
    raise AssertionError(f"Handshake written but assertions failed: {failed}")

print(f"Handshake -> {summary_path.relative_to(BASE)}")
print(f"All {len(summary['assertions'])} assertions passed.\n")
print(json.dumps({"status": summary["status"], **summary["assertions"]}, indent=2))

Handshake -> outputs/notebook_exports/summary_NB01.json
All 5 assertions passed.

{
  "status": "SUCCESS",
  "raw_shape_is_100000x27": true,
  "feature_shape_is_100000x7": true,
  "zero_nan_after_currency_strip": true,
  "all_features_float64": true,
  "parquet_roundtrip_verified": true
}


---
## NB01 Complete

| Check | Result |
|---|---|
| Raw shape | 100,000 × 27 |
| Feature matrix | 100,000 × 7, all `float64` |
| NaN after currency strip | **0** — assertion held |
| Data provenance | SHA-256 verified against the parity snapshot |

**Next:** return `summary_NB01.json`, `nb01_fig1_distributions.png` and
`nb01_fig2_correlation_heatmap.png` to the repository, then generate **NB02**.

NB02 must reproduce **LOF(auto) = 2,565**. That is the first genuinely diagnostic parity
anchor in the pipeline — the IF and LOF(0.05) counts of 5,000 are `contamination × n` and
would pass even on corrupted input.